In [2]:
import pandas as pd
import sys
from pathlib import Path
import subprocess

# Downloading dataset

In [3]:
RAW_DIR = Path("data/raw")
RAW_FILE = RAW_DIR / "accepted_2007_to_2018Q4.csv.gz"

RAW_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_FILE.exists():
    print("Downloading LendingClub dataset from Kaggle...")

    subprocess.run([
        sys.executable,
        "-m", "kaggle",
        "datasets",
        "download",
        "wordsforthewise/lending-club",
        "-f",
        "accepted_2007_to_2018Q4.csv.gz",
        "-p",
        str(RAW_DIR)
    ], check=True)

print("Dataset found.")
print("Starting preprocessing...")

Dataset found.
Starting preprocessing...


In [4]:
headers =  [    'id','loan_amnt',
                'term','int_rate',
                'installment','grade',
                'sub_grade','emp_length',
                'home_ownership','annual_inc',
                'verification_status','issue_d',
                'loan_status','purpose',
                'addr_state','dti',
                'delinq_2yrs','earliest_cr_line',
                'fico_range_low','fico_range_high',
                'inq_last_6mths','open_acc',
                'pub_rec','revol_bal',
                'revol_util','total_acc',
                'tot_cur_bal','bc_util',
                'mort_acc','pub_rec_bankruptcies'
            ]


for i, chunk in enumerate(pd.read_csv(RAW_FILE,
                  usecols=headers, low_memory=False, chunksize=500000)):
    print(f"Processing chunk {i+1}/5 with {len(chunk)} rows...")
    write_header = (i == 0)
    chunk.to_csv("data/dataset.csv", index=False, mode='a', header=write_header)

df = pd.read_csv('data/dataset.csv', low_memory=False)

print("Dataset loaded as csv. Starting cleaning and preprocessing...")

print("Initial dataset shape:", df.shape)

Processing chunk 1/5 with 500000 rows...
Processing chunk 2/5 with 500000 rows...
Processing chunk 3/5 with 500000 rows...
Processing chunk 4/5 with 500000 rows...
Processing chunk 5/5 with 260701 rows...
Dataset loaded as csv. Starting cleaning and preprocessing...
Initial dataset shape: (2260701, 30)


Imputing rows from a given subset

In [5]:
subset = [
            'id','loan_amnt',
            'term','int_rate',
            'installment','grade',
            'sub_grade','home_ownership',
            'annual_inc','verification_status',
            'loan_status','purpose',
            'addr_state','earliest_cr_line',
            'fico_range_low','fico_range_high'
]
before = len(df)
df.dropna(subset=subset, inplace=True)
after = len(df)
print(f"Dropped {before - after} rows with missing values in subset columns.")

Dropped 62 rows with missing values in subset columns.


Type cast and limit for 1-1-2015 to 31-12-2018 

In [6]:
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y")

In [7]:
df = df[
    (df["issue_d"] >= "2015-01-01") &
    (df["issue_d"] <= "2018-12-31")
]

In [8]:
df["earliest_cr_line"] = pd.to_datetime(
    df["earliest_cr_line"],
    format="%b-%Y",
    errors="coerce"
)

Final Cleanup

In [9]:
df['term'] = df['term'].str.replace(' months', '').astype(int)

In [12]:
df['credit_history_months'] = (
    (df["issue_d"] - df["earliest_cr_line"]).dt.days / 30.44
).round()
df.drop(columns=["earliest_cr_line"], inplace=True)

In [13]:
df["fico_score"] = (
    df["fico_range_low"] + df["fico_range_high"]
) / 2
df.drop(
    columns=["fico_range_low", "fico_range_high"],
    inplace=True
)

In [14]:
df.drop(columns=["grade"], inplace=True)

In [15]:
cols = ['mort_acc', 'pub_rec_bankruptcies', 'revol_util', 'dti']
for col in cols:
    df[col] = df[col].fillna(df[col].median())

In [16]:
valid_statuses = {
    "Charged Off": 1,
    "Default": 1,
    "Fully Paid": 0
}
df["default"] = df["loan_status"].map(valid_statuses)
# Dropping rows where 'loan_status' is not in valid_statuses
# i.e Current, In Grace Period, Late (16-30 days), Late (31-120 days), "does not meet credit policy"
df = df[df["default"].notna()].copy()
df["default"] = df["default"].astype(int)

In [18]:
for col in ['tot_cur_bal', 'bc_util']:
    null_rate = df[col].isnull().mean() * 100
    print(f"Null Rate for {col}: {null_rate:.2f}%")

Null Rate for tot_cur_bal: 0.00%
Null Rate for bc_util: 1.18%


In [ ]:
df['emp_length']
df['emp_length'] = (
    df['emp_length']
    .str.replace(' years', '')
    .str.replace(' year', '')
    .str.replace('< 1', '0')
    .str.replace('10+', '10')
    .fillna('-1')
    .astype(int)
)

0          10
1          10
2          10
4           3
5           4
           ..
2260688     5
2260690     9
2260691     3
2260692    10
2260697     6
Name: emp_length, Length: 894290, dtype: int64

In [ ]:
file_path = Path("processed/synthetic_lendingclub.csv")
file_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(file_path, index=False)

In [26]:
df.shape
print("\nClass Balance: ")
df['issue_year'] = df['issue_d'].dt.year

print(df['issue_year'].value_counts().sort_index())


Class Balance: 
issue_year
2015    375546
2016    293105
2017    169321
2018     56318
Name: count, dtype: int64


In [27]:
df.shape

(894290, 30)

# Employement length left cz this nga isnt replying